# Column with node-to-surface constraints — v4

Same model, solver setup, and base-fixity trick (`zeroLength` to ground) as v3. **New in v4**: after the static analysis, the displacement field is pushed into a `Results` object and visualized through apeGmsh's `Results.viewer()`.

## The result pipeline

```
OpenSees  →  numpy arrays  →  Results.from_fem(fem, point_data={...})  →  results.viewer()
```

Fields attached in this notebook:

| name | shape | description |
|------|-------|-------------|
| `Displacement` | `(N, 3)` | nodal displacement vector — the viewer auto-detects this for the deformed-shape toggle |
| `\|U\|`        | `(N,)`  | displacement magnitude for contour colouring |
| `Ux`, `Uy`, `Uz` | `(N,)` | translation components for individual contour views |

Node ordering follows `fem.nodes.ids` — `Results.from_fem` remaps everything to 0-based VTK indices internally, so we just iterate `fem.nodes.ids` in order and call `ops.nodeDisp(int(tag))` for each.

Note: `fem.nodes.ids` contains only the 646 solid/reference nodes — the 44 constraint-generated phantom nodes live under `fem.nodes.constraints` and are **not** in the viewer mesh. That's fine: they exist only to implement the constraint chain, and OpenSees handles their kinematics invisibly.

In [ ]:
from apeGmsh import apeGmsh, Part, Results
from apeGmsh import FEMData
from apeGmsh.sections import W_solid
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
m1 = apeGmsh(model_name='beam_column', verbose=False)
m1.begin()

column = m1.sections.W_solid(bf=150, tf=20, h=300, tw=10, length=2000, label='column')

m1.model.selection.select_volumes().to_physical(name='pg_column')

base = m1.model.geometry.add_point(x=0, y=0, z=0,    lc=100, label='base')
top  = m1.model.geometry.add_point(x=0, y=0, z=2000, lc=100, label='top')

m1.constraints.node_to_surface('base', column.labels.start_face)
m1.constraints.node_to_surface('top',  column.labels.end_face)

m1.loads.point(
    target='top',
    force_xyz=[0, 1000, 0],
    moment_xyz=[0, 0, 0],
)

m1.mesh.sizing.set_global_size(100)
m1.mesh.generation.generate(dim=3)

fem = m1.mesh.queries.get_fem_data(remove_orphans=True)

m1.model.viewer()
m1.mesh.viewer()

m1.end()

## OpenSees model + analysis (same as v3)

- `ndf = 6` global, solid column nodes overridden to `ndf = 3`.
- Grounded helper node at `GROUND_TAG` fixed in 6 DOFs, connected to the base reference point via a stiff 6-DOF `zeroLength` spring — so `nodeReaction` works directly on the ground node.
- `constraints('Penalty')` because the phantom nodes are daisy-chained (`rigidLink` slave + `equalDOF` master), which breaks `Transformation`.

In [ ]:
import openseespy.opensees as ops

ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 6)

ops.timeSeries('Linear', 1)

# --- Nodes -----------------------------------------------------------------
for nid, xyz in fem.nodes.get(target='pg_column'):
    ops.node(nid, *xyz, '-ndf', 3)

ref = fem.nodes.get(target=['base', 'top'])
for nid, xyz in ref:
    ops.node(nid, *xyz)

for nid, xyz in fem.nodes.constraints.phantom_nodes():
    ops.node(nid, *xyz)

# --- Grounded helper node + zeroLength spring at the base ------------------
base_id = int(next(iter(fem.nodes.get_ids(target='base'))))
base_xyz = next(xyz for nid, xyz in ref if nid == base_id)

GROUND_TAG = 10_000
ops.node(GROUND_TAG, *base_xyz)
ops.fix(GROUND_TAG, 1, 1, 1, 1, 1, 1)

K_SPRING = 1e14
for i in range(6):
    ops.uniaxialMaterial('Elastic', 100 + i, K_SPRING)

ZL_TAG = 20_000
ops.element('zeroLength', ZL_TAG, GROUND_TAG, base_id,
            '-mat', 100, 101, 102, 103, 104, 105,
            '-dir', 1, 2, 3, 4, 5, 6)

# --- Material + tet elements -----------------------------------------------
E  = 200e3
nu = 0.3
ops.nDMaterial('ElasticIsotropic', 1, E, nu)

for group in fem.elements.get(element_type='tet4'):
    for eid, conn in group:
        ops.element('FourNodeTetrahedron', int(eid), *[int(n) for n in conn], 1)

# --- MP constraints --------------------------------------------------------
for master, slaves in fem.nodes.constraints.rigid_link_groups():
    for slave in slaves:
        ops.rigidLink('beam', int(master), int(slave))

for pair in fem.nodes.constraints.equal_dofs():
    ops.equalDOF(int(pair.master_node), int(pair.slave_node), *pair.dofs)

# --- Loads -----------------------------------------------------------------
ops.pattern('Plain', 1, 1)
for load in fem.nodes.loads.by_kind('nodal'):
    fx, fy, fz = load.force_xyz  or (0.0, 0.0, 0.0)
    mx, my, mz = load.moment_xyz or (0.0, 0.0, 0.0)
    ops.load(int(load.node_id), fx, fy, fz, mx, my, mz)

# --- Analysis --------------------------------------------------------------
ops.constraints('Penalty', 1e15, 1e15)
ops.numberer('RCM')
ops.system('UmfPack')
ops.test('NormDispIncr', 1.0e-6, 10)
ops.algorithm('Linear')
ops.integrator('LoadControl', 0.1)
ops.analysis('Static')

for i in range(10):
    ok = ops.analyze(1)
    print(f'Step {i+1}: {"ok" if ok == 0 else f"failed ({ok})"}')

## Base reaction (via the grounded zeroLength)

Expected from equilibrium: `Fy = -1000 N`, `Mx = +2·10⁶ N·mm`.

In [ ]:
ops.reactions()

rxn = ops.nodeReaction(GROUND_TAG)

print(f'nodeReaction(ground = {GROUND_TAG}):')
print(f'  Fx = {rxn[0]:+.4e}   Fy = {rxn[1]:+.4e}   Fz = {rxn[2]:+.4e}')
print(f'  Mx = {rxn[3]:+.4e}   My = {rxn[4]:+.4e}   Mz = {rxn[5]:+.4e}')
print(f'\nExpected:  Fy = -1.0000e+03   Mx = +2.0000e+06')

## Extract the displacement field

Build an `(N, 3)` array of displacements following the `fem.nodes.ids` order. For nodes that were created with `-ndf 3` (the solid column nodes) `ops.nodeDisp` returns 3 values; for the 6-DOF reference points we only keep the translation components. `Results.from_fem` consumes the array as-is.

In [ ]:
n_nodes = len(fem.nodes.ids)
disp = np.zeros((n_nodes, 3), dtype=np.float64)

for i, nid in enumerate(fem.nodes.ids):
    d = ops.nodeDisp(int(nid))
    disp[i, 0] = d[0]
    disp[i, 1] = d[1]
    disp[i, 2] = d[2]

u_mag = np.linalg.norm(disp, axis=1)

print(f'Extracted displacements for {n_nodes} nodes.')
print(f'  ux range: [{disp[:,0].min():+.4e}, {disp[:,0].max():+.4e}]')
print(f'  uy range: [{disp[:,1].min():+.4e}, {disp[:,1].max():+.4e}]')
print(f'  uz range: [{disp[:,2].min():+.4e}, {disp[:,2].max():+.4e}]')
print(f'  |U| max:  {u_mag.max():+.4e}')

## Build and launch the Results viewer

`Results.from_fem` wraps the mesh geometry from `fem` together with the numpy fields into a self-contained, session-free object. Calling `results.viewer()` pops the interactive apeGmsh Qt/VTK viewer — use the **Deformation** controls to toggle the deformed shape and the **Contour** dropdown to switch between `|U|`, `Ux`, `Uy`, `Uz`, or the raw `Displacement` vector magnitude.

`blocking=False` runs the viewer in a subprocess so the notebook remains interactive.

In [ ]:
results = Results.from_fem(
    fem,
    point_data={
        'Displacement': disp,
        '|U|':          u_mag,
        'Ux':           disp[:, 0],
        'Uy':           disp[:, 1],
        'Uz':           disp[:, 2],
    },
    name='column_nts_v4',
)

results

In [ ]:
results.viewer(blocking=False)

## Cantilever closed-form sanity check

Strong-axis bending of the W section (`Ix = bf·H³/12 − (bf − tw)·h³/12`, `H = 340 mm`):

- `δ_EB = P·L³ / (3·E·Ix)`  (Euler-Bernoulli)
- `δ_T  = δ_EB + P·L / (G·As)`, `As ≈ tw·h`  (Timoshenko, adds shear)

In [ ]:
top_id = int(next(iter(fem.nodes.get_ids(target='top'))))
tip_disp = ops.nodeDisp(top_id)

bf, tf, h_web, tw, L = 150.0, 20.0, 300.0, 10.0, 2000.0
H  = 2 * tf + h_web
Ix = (bf * H**3) / 12 - ((bf - tw) * h_web**3) / 12
P  = 1000.0

delta_EB    = P * L**3 / (3 * E * Ix)
G           = E / (2 * (1 + nu))
As          = tw * h_web
delta_shear = P * L / (G * As)
delta_T     = delta_EB + delta_shear

print(f'Top node {top_id} displacement:')
print(f'  ux = {tip_disp[0]:+.4e}   uy = {tip_disp[1]:+.4e}   uz = {tip_disp[2]:+.4e}')
print(f'\nCantilever closed-form:')
print(f'  delta_EB         = {delta_EB:.4e} mm')
print(f'  delta_Timoshenko = {delta_T:.4e} mm')
print(f'  FEM uy           = {tip_disp[1]:+.4e} mm')
print(f'  FEM / EB         = {tip_disp[1] / delta_EB:.4f}')
print(f'  FEM / Timoshenko = {tip_disp[1] / delta_T:.4f}')